# Mean-Field Game Fixed-Point Solver

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drsamirasaf-creator/ge-lav-companion-code/blob/main/notebooks/05_mfg_solver.ipynb)

> **Colab users:** the first cell auto-installs `gelav`. Local users can skip it.

*Course sessions: S21 (McKean-Vlasov & MFG), S26 (MFG existence proofs)*

---

The GE-LAV equilibrium is the fixed point of a coupled system:

- **Backward HJB** — each LP's optimal value function V(L,t), given the
  population's distribution μ
- **Forward Fokker-Planck** — the evolution of μ, given everyone's optimal
  policy α*(L,t)

The two equations are coupled: V depends on μ (through the secondary market
price π(L,T;μ)), and μ depends on V (through α*). This notebook implements
the standard MFG fixed-point iteration that finds (V*, μ*) jointly.

In [ ]:
# Auto-install gelav if running in Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/drsamirasaf-creator/ge-lav-companion-code.git

## 1. Setup

We solve a Mean-Field Game where each LP's running reward and exit value
both depend on the population state μ. Concretely, the secondary market
price is:

$$\pi(L, T-t; \mu) = \alpha(\mu) + \beta(\mu) \cdot L + \gamma(\mu) \cdot L^2$$

where the coefficients depend (weakly) on the cross-sectional mean of μ
to model the externality. The simplest such coupling is:

$$\alpha(\mu) = \alpha_0 - \lambda \cdot E_\mu[L]$$

so when many LPs are in low-L states, secondary prices fall (the externality).
The course-calibrated coupling strength is λ ≈ 0.5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from gelav.mfg import mfg_solver
from gelav.term_structure import pi_of_L_T

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'
NAVY = '#1E3A5F'; GOLD = '#C89B3C'; RED = '#C0392B'

## 2. Run the fixed-point iteration

`mfg_solver` does the iteration:
1. Start with an initial guess μ₀ (Gaussian centered at L̄)
2. Solve HJB given μ₀ → get α*(L,t)
3. Solve Fokker-Planck given α* → get new μ₁
4. Check ||μ₁ - μ₀|| < tol. If not, repeat.

Convergence is typically achieved in 5-15 iterations for our calibration.

In [ ]:
result = mfg_solver(
    kappa=0.45,
    sigma=0.32,
    L_bar=1.0,
    r=0.08,
    T_fund=10.0,
    coupling_strength=0.5,
    n_L=101,
    n_t=100,
    max_iter=30,
    tol=1e-3,
    verbose=True,
)
print('\nKeys in result:', list(result.keys()))
print(f'Converged: {result.get("converged")}')
print(f'Iterations: {result.get("n_iter")}')

## 3. Visualize the equilibrium V(L, t)

The value function V(L,t) is high in the upper-right (good state, lots of
time remaining) and falls along the exit boundary. The contour plot below
shows V(L,t) on a heatmap.

In [ ]:
V = result['V']  # shape (n_L, n_t)
L_grid = result['L_grid']
t_grid = result['t_grid']

fig, ax = plt.subplots(figsize=(9, 5))
cs = ax.contourf(t_grid, L_grid, V, levels=20, cmap='RdYlGn')
plt.colorbar(cs, ax=ax, label='V(L, t)')
ax.set_xlabel('t (time elapsed, years)')
ax.set_ylabel('L (liquidity state)')
ax.set_title('MFG equilibrium value function V(L, t)', color=NAVY, weight='bold')
ax.axhline(y=1.0, color='white', linestyle='--', alpha=0.5, label='L̄ = 1.0')
ax.legend()
plt.tight_layout(); plt.show()

## 4. Equilibrium distribution μ(L, t)

The cross-sectional density of LPs across L states, evolved by the
Fokker-Planck equation under everyone's optimal policy. Starts concentrated
at L=L̄ and spreads over time.

In [ ]:
mu = result['mu']  # shape (n_L,) — stationary cross-sectional density

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(L_grid, mu, color=NAVY, lw=2.5)
ax.fill_between(L_grid, mu, alpha=0.3, color=NAVY)
# Compare to OU stationary distribution (which is Normal(L_bar, sigma^2/(2*kappa)))
from scipy.stats import norm
stat_std = 0.32 / np.sqrt(2 * 0.45)
ou_stat = norm.pdf(L_grid, loc=1.0, scale=stat_std)
ax.plot(L_grid, ou_stat, '--', color=RED, lw=2, label='OU stationary (no coupling)')
ax.set_xlabel('L'); ax.set_ylabel('μ(L) — equilibrium density')
ax.set_xlim(-2, 4); ax.grid(alpha=0.3); ax.legend()
ax.set_title('Equilibrium LP-state distribution (stationary)', color=NAVY, weight='bold')
plt.tight_layout(); plt.show()

## 5. Convergence history

How quickly did the fixed-point iteration converge? Plot the residual
||μ_{n+1} - μ_n|| as a function of iteration.

In [ ]:
history = result.get('history', [])
if history:
    # history is a list of dicts: [{'iteration': i, 'mu_diff': ..., 'E_L': ...}, ...]
    residuals = np.array([h['mu_diff'] for h in history])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.semilogy(range(1, len(residuals)+1), residuals, '-o',
                color=GOLD, lw=2, markersize=6)
    ax.axhline(y=1e-3, color=RED, linestyle='--', alpha=0.7, label='tolerance')
    ax.set_xlabel('Iteration'); ax.set_ylabel('||μ_n+1 − μ_n||  (log)')
    ax.set_title('MFG fixed-point convergence', color=NAVY, weight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    
    # Also print the iteration log
    import pandas as pd
    pd.DataFrame(history).style.format({
        'E_L': '{:.4f}', 'alpha_eff': '{:.4f}', 'mu_diff': '{:.2e}'
    })
else:
    print('History not tracked in this build.')

## 6. Effect of coupling strength

When λ (`coupling_strength`) is 0, agents act independently — MFG reduces
to a non-game stochastic control problem. As λ increases, the externality
matters more: each LP's exit affects everyone's prices.

In [ ]:
# Re-run with several coupling strengths and compare equilibrium L*(t) curves
couplings = [0.0, 0.25, 0.5, 1.0]
fig, ax = plt.subplots(figsize=(9, 5))

for lam in couplings:
    res = mfg_solver(kappa=0.45, sigma=0.32, L_bar=1.0, r=0.08, T_fund=10.0,
                     coupling_strength=lam, n_L=81, n_t=80, max_iter=15, tol=1e-3,
                     verbose=False)
    ax.plot(res['t_grid'], res['L_star'], lw=2, label=f'λ = {lam}')

ax.set_xlabel('t (years elapsed)'); ax.set_ylabel('L*(t)')
ax.set_title('Exit boundary L*(t) under different coupling strengths',
             color=NAVY, weight='bold')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='L̄ = 1.0')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Suggested exercises

1. **(easy)** Re-run with `kappa=0.9` (faster reversion). How does V(L,t) change?
   Where is the equilibrium density most concentrated?

2. **(medium)** Add a custom coupling: make β a function of the variance of μ
   (so dispersion among LPs affects prices, not just the mean).
   Modify `mfg_solver` accordingly and report whether convergence
   still holds.

3. **(hard)** Implement the heterogeneous-agent extension: two types of LPs
   (pension vs. endowment) with different risk aversions. Use a 2D μ
   indexed by (L, type). Identify what changes about the equilibrium L*(t).